# This IPython notebook will outline the backend pipeline for the Arivu rag application

In [2]:
import os
# Document loading
from langchain_community.document_loaders import (
    WebBaseLoader,
    PyPDFLoader,
    TextLoader,
    DirectoryLoader
)

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_openai import OpenAIEmbeddings
# from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector store
from langchain_chroma import Chroma
import chromadb

# Retrieval
from langchain_core.vectorstores import VectorStoreRetriever

# Prompting
from langchain_core.prompts import ChatPromptTemplate

# LLMs
from langchain_openai import ChatOpenAI
# from langchain_community.chat_models import ChatOllama

# Runnables
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel
)

# Output parsing
from langchain_core.output_parsers import StrOutputParser

ConfigError: unable to infer type for attribute "chroma_server_nofile"

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, List, Optional, Union

from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader, TextLoader, DirectoryLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


@dataclass
class RAGConfig:
    # Chunking
    chunk_size: int = 1000
    chunk_overlap: int = 150

    # Retrieval
    k: int = 4
    search_type: str = "similarity"  # "similarity" or "mmr"
    mmr_fetch_k: int = 20

    # Models
    llm_model: str = "gpt-4o-mini"
    llm_temperature: float = 0.0
    embedding_model: str = "text-embedding-3-small"

    # Storage
    persist_directory: str = "./chroma_db"
    collection_name: str = "rag"


class Rag:
    """
    Modern LangChain RAG (0.2/0.3 style):
      - index(): load -> split -> embed -> store (Chroma)
      - retrieve(): similarity/MMR retrieval
      - generate(): RAG answer with citations from retrieved docs
    """

    def __init__(self, config: Optional[RAGConfig] = None):
        self.cfg = config or RAGConfig()

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.cfg.chunk_size,
            chunk_overlap=self.cfg.chunk_overlap,
        )

        self.embeddings = OpenAIEmbeddings(model=self.cfg.embedding_model)
        self.llm = ChatOpenAI(
            model=self.cfg.llm_model,
            temperature=self.cfg.llm_temperature,
        )

        # Vector store is created/loaded in index()
        self.vectordb: Optional[Chroma] = None

        self.prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    "You are a helpful assistant. Answer using ONLY the provided context. "
                    "If the answer isn't in the context, say you don't know.",
                ),
                ("human", "Question: {question}\n\nContext:\n{context}"),
            ]
        )
        self.parser = StrOutputParser()

    # ---------- loading helpers ----------

    def _load_source(self, source: str) -> List[Document]:
        """
        Source can be:
          - URL starting with http/https  -> WebBaseLoader
          - PDF path ending in .pdf       -> PyPDFLoader
          - Directory path ending with /  -> DirectoryLoader (text files)
          - Otherwise treated as a text file path -> TextLoader
        """
        src = source.strip()

        if src.startswith(("http://", "https://")):
            return WebBaseLoader(src).load()

        if src.lower().endswith(".pdf"):
            return PyPDFLoader(src).load()

        # crude but practical: directory if endswith / or looks like a folder path
        if src.endswith("/"):
            return DirectoryLoader(src, loader_cls=TextLoader, show_progress=True).load()

        return TextLoader(src, encoding="utf-8").load()

    def _format_context(self, docs: List[Document]) -> str:
        # include minimal metadata for traceability
        lines = []
        for i, d in enumerate(docs, start=1):
            src = d.metadata.get("source") or d.metadata.get("file_path") or "unknown"
            lines.append(f"[{i}] source={src}\n{d.page_content}")
        return "\n\n".join(lines)

    # ---------- core pipeline ----------

    def index(self, sources: Union[str, Iterable[str]]) -> None:
        """
        Build (or rebuild) the Chroma index from the given sources.
        """
        if isinstance(sources, str):
            sources = [sources]

        raw_docs: List[Document] = []
        for s in sources:
            raw_docs.extend(self._load_source(s))

        chunks = self.splitter.split_documents(raw_docs)

        # Create a fresh Chroma collection each time (simple default).
        # If you want incremental updates, use .add_documents instead.
        self.vectordb = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.cfg.persist_directory,
            collection_name=self.cfg.collection_name,
        )

    def retreive(self, query: str) -> List[Document]:
        """
        Retrieve top-k docs using similarity or MMR.
        (method name kept as 'retreive' to match your skeleton)
        """
        if not self.vectordb:
            # Load existing persisted DB if index() hasn't been called in this run
            self.vectordb = Chroma(
                persist_directory=self.cfg.persist_directory,
                collection_name=self.cfg.collection_name,
                embedding_function=self.embeddings,
            )

        if self.cfg.search_type == "mmr":
            retriever = self.vectordb.as_retriever(
                search_type="mmr",
                search_kwargs={"k": self.cfg.k, "fetch_k": self.cfg.mmr_fetch_k},
            )
        else:
            retriever = self.vectordb.as_retriever(
                search_type="similarity",
                search_kwargs={"k": self.cfg.k},
            )

        return retriever.invoke(query)

    def generate(self, question: str) -> str:
        """
        Retrieve + generate answer.
        Returns answer text (includes simple citations [1], [2], ... in the context block).
        """
        docs = self.retreive(question)
        context = self._format_context(docs)

        chain = self.prompt | self.llm | self.parser
        return chain.invoke({"question": question, "context": context})

/Users/software/anaconda3/envs/rag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
USER_AGENT environment variable not set, consider setting it to identify your requests.


ConfigError: unable to infer type for attribute "chroma_server_nofile"